# =============================================================================
# PROJETO: MNIST Digit Classifier - Análise Preditiva Multiclasse
# AUTOR: Guillermo Jose Salas Avila
# DATA: 10/09/2026
# DESCRIÇÃO: Pipeline completo de Machine Learning para classificação de 
#            dígitos manuscritos usando o dataset MNIST.
#            Modelos: Random Forest, KNN e MLP (scikit-learn)
# =============================================================================

# 0- Importação de bibliotecas

In [ ]:
# -----------------------------------------------------------------------------
# BIBLIOTECAS CORE
# -----------------------------------------------------------------------------
import numpy as np              # Operações matemáticas e arrays
import pandas as pd             # Manipulação de dados tabulares
import matplotlib.pyplot as plt # Visualizações
import seaborn as sns           # Visualizações estatísticas
import time                     # Medição de tempo
import os                       # Manipulação de pastas e arquivos
import warnings
warnings.filterwarnings('ignore')

# -----------------------------------------------------------------------------
# MACHINE LEARNING (Scikit-Learn)
# -----------------------------------------------------------------------------
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# -----------------------------------------------------------------------------
# PROCESSAMENTO DE IMAGENS 
# -----------------------------------------------------------------------------
from PIL import Image, ImageOps

# -----------------------------------------------------------------------------
# CONFIGURAÇÕES
# -----------------------------------------------------------------------------
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("=" * 80)
print("🧠 MNIST - ANÁLISE PREDITIVA MULTICLASSE")
print("=" * 80)
print("✅ Importações realizadas com sucesso!")
print("📌 Modelos: Random Forest, KNN, MLP (scikit-learn)")
print("=" * 80)


# =============================================================================
# CRIAÇÃO AUTOMÁTICA DA ESTRUTURA DE PASTAS
# =============================================================================

def criar_estrutura_pastas():
    """
    Cria toda a estrutura de pastas necessária para o projeto.
    Garante que o projeto funcione em qualquer ambiente sem erros de
    'pasta não encontrada'.
    """
    estrutura = {
        'data': 'Dados do projeto (MNIST)',
        'data/own_images': 'Imagens manuscritas próprias (Desafio C)',
        'results': 'Gráficos, matrizes e resultados',
        'docs': 'Documentação adicional',
        'docs/images': 'Imagens usadas no README',
    }
    
    print("\n" + "=" * 80)
    print("📁 CRIANDO ESTRUTURA DE PASTAS")
    print("=" * 80)
    
    for pasta, descricao in estrutura.items():
        try:
            if os.path.exists(pasta):
                print(f"✅ Já existe: {pasta}/  ({descricao})")
            else:
                os.makedirs(pasta, exist_ok=True)
                print(f"🆕 Criada:   {pasta}/  ({descricao})")
        except Exception as e:
            print(f"❌ Erro ao criar {pasta}: {e}")
    
    print("=" * 80)
    print("✅ ESTRUTURA PRONTA!")
    print("=" * 80)

# Executar a criação das pastas
criar_estrutura_pastas()

# Verificar se há imagens próprias para o Desafio C
own_images_path = 'data/own_images'
if os.path.exists(own_images_path):
    imagens_encontradas = [f for f in os.listdir(own_images_path)
                          if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if imagens_encontradas:
        print(f"\n📸 {len(imagens_encontradas)} imagem(ns) encontrada(s) em data/own_images/")
        for img in sorted(imagens_encontradas):
            print(f"   → {img}")
    else:
        print("\n⚠️ Pasta data/own_images/ está vazia.")
        print("💡 Adicione imagens no formato digit_X.jpg (X = dígito 0-9)")

# 1- Carregamento e Análise Exploratória (EDA)

In [ ]:
# =============================================================================
# ETAPA 1: CARREGAMENTO E ANÁLISE EXPLORATÓRIA (EDA)
# =============================================================================

print("\n" + "=" * 80)
print("📥 ETAPA 1: CARREGAMENTO DO DATASET MNIST")
print("=" * 80)

# -----------------------------------------------------------------------------
# CAMINHOS DOS ARQUIVOS LOCAIS
# -----------------------------------------------------------------------------
# Definir onde salvar os dados em formato .npz (NumPy comprimido)
DATA_DIR = 'data'
MNIST_FILE = os.path.join(DATA_DIR, 'mnist.npz')

# Garantir que a pasta data/ existe
os.makedirs(DATA_DIR, exist_ok=True)


# -----------------------------------------------------------------------------
# FUNÇÃO PARA CARREGAR O MNIST (COM CACHE LOCAL)
# -----------------------------------------------------------------------------
def load_mnist_data(force_download=False):
    """
    Carrega o dataset MNIST com cache local.
    
    Estratégia:
    1. Se o arquivo local existir (e force_download=False), carrega do disco
    2. Caso contrário, baixa via fetch_openml e salva localmente
    
    Parâmetros:
    - force_download: se True, força o download mesmo se o arquivo local existir
    
    Retorna:
    - X: array de features (70000 × 784)
    - y: array de rótulos (70000,)
    """
    
    # -------------------------------------------------------------------------
    # CASO 1: Arquivo local existe e não queremos forçar download
    # -------------------------------------------------------------------------
    if os.path.exists(MNIST_FILE) and not force_download:
        print(f"\n💾 Arquivo local encontrado: {MNIST_FILE}")
        print("📂 Carregando dataset do disco (rápido)...")
        
        try:
            with np.load(MNIST_FILE) as data:
                X = data['X']
                y = data['y']
            print("✅ Dataset carregado do disco com sucesso!")
            return X, y
        except Exception as e:
            print(f"⚠️ Erro ao carregar arquivo local: {e}")
            print("📥 Baixando novamente...")
    
    # -------------------------------------------------------------------------
    # CASO 2: Arquivo local não existe (ou force_download=True) → baixar
    # -------------------------------------------------------------------------
    print("\n📥 Baixando dataset MNIST via fetch_openml...")
    print("(Pode levar alguns minutos na primeira execução)")
    
    # Baixar o dataset
    mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
    
    # Separar features e rótulos
    X = mnist.data.astype(np.float32)   # Converter para float32 (mais leve)
    y = mnist.target.astype(int)         # Converter rótulos para inteiro
    
    print("✅ Dataset baixado com sucesso!")
    
    # -------------------------------------------------------------------------
    # SALVAR LOCALMENTE EM FORMATO .npz (comprimido)
    # -------------------------------------------------------------------------
    print(f"\n💾 Salvando dataset em: {MNIST_FILE}")
    try:
        np.savez_compressed(MNIST_FILE, X=X, y=y)
        file_size_mb = os.path.getsize(MNIST_FILE) / (1024 * 1024)
        print(f"✅ Dataset salvo com sucesso! ({file_size_mb:.1f} MB)")
    except Exception as e:
        print(f"⚠️ Erro ao salvar arquivo: {e}")
    
    return X, y


# -----------------------------------------------------------------------------
# CARREGAR O DATASET
# -----------------------------------------------------------------------------
X, y = load_mnist_data()

# -----------------------------------------------------------------------------
# ANÁLISE DA DIMENSIONALIDADE
# -----------------------------------------------------------------------------
print("\n" + "=" * 60)
print("📊 DIMENSIONALIDADE DOS DADOS")
print("=" * 60)
print(f"X shape: {X.shape}")
print(f"  → {X.shape[0]} imagens (amostras)")
print(f"  → {X.shape[1]} features (28×28 = 784 pixels)")

print(f"\ny shape: {y.shape}")
print(f"  → {y.shape[0]} rótulos")
print(f"  → Classes: {np.unique(y)}")
print(f"  → Número de classes: {len(np.unique(y))}")

# Verificar valores dos pixels
print(f"\n🎨 VALORES DOS PIXELS:")
print(f"  Mínimo: {X.min()}")
print(f"  Máximo: {X.max()}")
print(f"  Média:  {X.mean():.2f}")
print(f"  Tipo:   {X.dtype}")
print(f"  Escala: 0 (preto) a 255 (branco)")

# -----------------------------------------------------------------------------
# VERIFICAR ARQUIVO SALVO
# -----------------------------------------------------------------------------
if os.path.exists(MNIST_FILE):
    size_mb = os.path.getsize(MNIST_FILE) / (1024 * 1024)
    print(f"\n📁 ARQUIVO SALVO:")
    print(f"  Caminho: {MNIST_FILE}")
    print(f"  Tamanho: {size_mb:.2f} MB")
    print(f"  Formato: .npz (NumPy comprimido)")
    print(f"\n💡 Nas próximas execuções, o dataset será carregado do disco (mais rápido)!")

In [ ]:
# -----------------------------------------------------------------------------
# DISTRIBUIÇÃO DAS CLASSES (BALANCEAMENTO)
# -----------------------------------------------------------------------------

print("\n📊 DISTRIBUIÇÃO DAS CLASSES:")
print("-" * 60)

# Contar imagens por dígito
unique, counts = np.unique(y, return_counts=True)
total = len(y)
min_count, max_count = counts.min(), counts.max()

for digit, count in zip(unique, counts):
    percentage = (count / total) * 100
    bar = "█" * int(percentage / 2)
    print(f"Dígito {digit}: {count:5d} imagens ({percentage:5.2f}%) {bar}")

print("-" * 60)
print(f"Total: {total}  |  Razão maior/menor: {max_count/min_count:.2f}")
print(f"✅ Dataset BALANCEADO")

# -----------------------------------------------------------------------------
# VISUALIZAÇÃO
# -----------------------------------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico de barras
bars = axes[0].bar(unique, counts, color='skyblue', edgecolor='navy', linewidth=1.5)
axes[0].set_xlabel('Dígito'); axes[0].set_ylabel('Número de Amostras')
axes[0].set_title('Distribuição das Classes no MNIST', fontweight='bold')
axes[0].set_xticks(range(10)); axes[0].grid(axis='y', alpha=0.3)
for bar, count in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height(),
                 f'{count}', ha='center', va='bottom', fontsize=10)

# Gráfico de pizza
colors = plt.cm.tab10(np.linspace(0, 1, 10))
axes[1].pie(counts, labels=unique, autopct='%1.1f%%', startangle=90,
            colors=colors, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Proporção das Classes', fontweight='bold')

plt.suptitle('Análise de Balanceamento do Dataset MNIST', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('results/class_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# -----------------------------------------------------------------------------
# GRADE VISUAL DE EXEMPLOS (2 × 5)
# -----------------------------------------------------------------------------

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
axes = axes.ravel()

for digit in range(10):
    idx = np.where(y == digit)[0][0]
    image = X[idx].reshape(28, 28)
    axes[digit].imshow(image, cmap='gray')
    axes[digit].set_title(f'Dígito: {digit}', fontsize=14, fontweight='bold')
    axes[digit].axis('off')

plt.suptitle('Exemplos de Cada Dígito do Dataset MNIST', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('results/digit_grid.png', dpi=300, bbox_inches='tight')
plt.show()

# -----------------------------------------------------------------------------
# INTERPRETAÇÃO DA ESTRUTURA
# -----------------------------------------------------------------------------

print("""
📖 INTERPRETAÇÃO DOS DADOS:

1. ESCALA DE INTENSIDADE (0 a 255):
   • 0 = Preto (fundo) | 255 = Branco (traço)

2. REPRESENTAÇÃO 2D (28 × 28 pixels):
   • Cada imagem é uma matriz de 28 linhas × 28 colunas

3. REPRESENTAÇÃO VETORIAL (784 features):
   • Cada imagem vira um vetor de 784 posições
   • Permite uso de algoritmos de ML tradicionais
""")

# 2- Pré-processamento e Divisão dos Dados

In [ ]:
# =============================================================================
# ETAPA 2: PRÉ-PROCESSAMENTO E DIVISÃO DOS DADOS
# =============================================================================

print("\n" + "=" * 80)
print("⚙️ ETAPA 2: PRÉ-PROCESSAMENTO")
print("=" * 80)

# Divisão estratificada: 70% treino / 10% validação / 20% teste
print("\n📊 Realizando divisão estratificada...")

# Passo 1: 80% (treino+val) e 20% (teste)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# Passo 2: 87.5% (treino) e 12.5% (val) do subconjunto → 70% e 10% do total
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.125,
    random_state=RANDOM_STATE, stratify=y_train_val
)

# Verificação dos tamanhos
print(f"\n📏 TAMANHO DOS CONJUNTOS:")
print("-" * 50)
print(f"Treino:    {X_train.shape[0]:5d} imagens ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Validação: {X_val.shape[0]:5d} imagens ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Teste:     {X_test.shape[0]:5d} imagens ({X_test.shape[0]/len(X)*100:.1f}%)")

# Verificar estratificação
print(f"\n🔍 Proporção das classes:")
print("-" * 50)
for digit in range(10):
    tr = (y_train == digit).sum() / len(y_train) * 100
    vl = (y_val == digit).sum() / len(y_val) * 100
    ts = (y_test == digit).sum() / len(y_test) * 100
    print(f"Dígito {digit}: Treino {tr:.2f}% | Val {vl:.2f}% | Teste {ts:.2f}%")
print("✅ Estratificação OK")

In [ ]:
# -----------------------------------------------------------------------------
# NORMALIZAÇÃO DOS PIXELS PARA [0.0, 1.0]
# -----------------------------------------------------------------------------

print("\n📏 Aplicando normalização dos pixels...")

X_train_norm = X_train / 255.0
X_val_norm   = X_val / 255.0
X_test_norm  = X_test / 255.0

print(f"\n✅ NORMALIZAÇÃO APLICADA:")
print(f"Min: {X_train_norm.min():.4f} | Max: {X_train_norm.max():.4f} | Média: {X_train_norm.mean():.4f}")

# -----------------------------------------------------------------------------
# JUSTIFICATIVA
# -----------------------------------------------------------------------------

print("""
📖 POR QUE NORMALIZAR?

1. CONVERGÊNCIA DE MODELOS LINEARES:
   • Gradientes mais estáveis

2. DISTÂNCIAS MÉTRICAS (KNN):
   • Evita que features de maior magnitude dominem

3. REDES NEURAIS (MLP):
   • Acelera convergência, evita saturação

4. COMPARAÇÃO JUSTA:
   • Todos os modelos sob as mesmas condições
""")

# 3- Implementação e Treinamento dos 3 Modelos

In [ ]:
# =============================================================================
# ETAPA 3: IMPLEMENTAÇÃO E TREINAMENTO DOS MODELOS
# =============================================================================

print("\n" + "=" * 80)
print("🤖 ETAPA 3: TREINAMENTO DOS MODELOS")
print("=" * 80)

models = {}
train_times = {}

# -----------------------------------------------------------------------------
# MODELO 1: RANDOM FOREST
# -----------------------------------------------------------------------------
print("\n" + "=" * 60)
print("🌲 MODELO 1: RANDOM FOREST")
print("=" * 60)

rf_model = RandomForestClassifier(
    n_estimators=100,       # 100 árvores
    max_depth=20,           # profundidade máxima (evita overfitting)
    min_samples_split=10,   # mínimo de amostras para dividir nó
    random_state=RANDOM_STATE,
    n_jobs=-1               # paralelização
)

start_time = time.time()
rf_model.fit(X_train_norm, y_train)
rf_train_time = time.time() - start_time
train_times['Random Forest'] = rf_train_time

print(f"✅ Treinado em {rf_train_time:.2f} segundos")
models['Random Forest'] = rf_model

In [ ]:
# -----------------------------------------------------------------------------
# MODELO 2: K-NEAREST NEIGHBORS (KNN)
# -----------------------------------------------------------------------------
print("\n" + "=" * 60)
print("👥 MODELO 2: K-NEAREST NEIGHBORS (KNN)")
print("=" * 60)

knn_model = KNeighborsClassifier(
    n_neighbors=5,       # 5 vizinhos
    weights='distance',  # peso por distância
    metric='euclidean',  # distância Euclidiana
    n_jobs=-1
)

start_time = time.time()
knn_model.fit(X_train_norm, y_train)
knn_train_time = time.time() - start_time
train_times['KNN'] = knn_train_time

print(f"✅ Treinado em {knn_train_time:.2f} segundos")
models['KNN'] = knn_model

In [ ]:
# -----------------------------------------------------------------------------
# MODELO 3: REDE NEURAL MLP (scikit-learn)
# -----------------------------------------------------------------------------
print("\n" + "=" * 60)
print("🧠 MODELO 3: REDE NEURAL MLP (scikit-learn)")
print("=" * 60)

mlp_model = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64),  # 3 camadas ocultas
    activation='relu',                   # função de ativação
    solver='adam',                       # otimizador adaptativo
    learning_rate_init=0.001,            # taxa de aprendizado
    max_iter=50,                         # máximo de épocas
    early_stopping=True,                 # parada antecipada
    validation_fraction=0.1,             # 10% para validação interna
    alpha=0.0001,                        # regularização L2
    random_state=RANDOM_STATE,
    verbose=False
)

start_time = time.time()
mlp_model.fit(X_train_norm, y_train)
mlp_train_time = time.time() - start_time
train_times['MLP (scikit-learn)'] = mlp_train_time

print(f"✅ Treinado em {mlp_train_time:.2f} segundos")
print(f"   Iterações: {mlp_model.n_iter_}")
models['MLP (scikit-learn)'] = mlp_model

# -----------------------------------------------------------------------------
# CURVA DE APRENDIZADO DO MLP
# -----------------------------------------------------------------------------

plt.figure(figsize=(10, 6))
plt.plot(mlp_model.loss_curve_, linewidth=2, color='darkblue')
plt.xlabel('Iteração'); plt.ylabel('Loss')
plt.title('MLP - Curva de Aprendizado', fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('results/mlp_learning_curve.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n" + "=" * 60)
print(f"✅ TODOS OS {len(models)} MODELOS TREINADOS!")
print("=" * 60)

# 4- Avaliação Comparativa

In [ ]:
# =============================================================================
# ETAPA 4.1 — Função de Avaliação
# =============================================================================

print("\n" + "=" * 80)
print("📊 ETAPA 4: AVALIAÇÃO COMPARATIVA")
print("=" * 80)

def evaluate_model(model, X_test, y_test, model_name):
    """Avalia modelo e retorna métricas + matriz de confusão."""
    print(f"\n{'='*60}\n🔍 AVALIAÇÃO: {model_name}\n{'='*60}")
    
    start_time = time.time()
    y_pred = model.predict(X_test)
    pred_time = time.time() - start_time
    
    # Métricas
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec  = recall_score(y_test, y_pred, average='weighted')
    f1   = f1_score(y_test, y_pred, average='weighted')
    
    print(f"\n📈 MÉTRICAS:")
    print(f"  Acurácia:            {acc:.4f} ({acc*100:.2f}%)")
    print(f"  Precisão (weighted): {prec:.4f}")
    print(f"  Recall (weighted):   {rec:.4f}")
    print(f"  F1-Score (weighted): {f1:.4f}")
    print(f"  Tempo de predição:   {pred_time:.4f} s")
    
    # Matriz de confusão
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=range(10), yticklabels=range(10),
                cbar_kws={'label': 'Contagem'})
    plt.title(f'Matriz de Confusão - {model_name}', fontweight='bold')
    plt.xlabel('Predito'); plt.ylabel('Real')
    plt.tight_layout()
    safe = model_name.replace(" ", "_").replace("(", "").replace(")", "")
    plt.savefig(f'results/confusion_matrix_{safe}.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Classification Report
    print(f"\n📋 Classification Report:")
    print(classification_report(y_test, y_pred, digits=4))
    
    return {'model': model_name, 'accuracy': acc, 'precision': prec,
            'recall': rec, 'f1': f1, 'pred_time': pred_time,
            'cm': cm, 'y_pred': y_pred}

In [ ]:
# -----------------------------------------------------------------------------
# 4.2 — Avaliar Todos os Modelos
# -----------------------------------------------------------------------------

results = []
for model_name, model in models.items():
    result = evaluate_model(model, X_test_norm, y_test, model_name)
    result['train_time'] = train_times.get(model_name, 0)
    results.append(result)

print("\n✅ Todos os modelos avaliados!")

In [ ]:
# -----------------------------------------------------------------------------
# 4.3 — Tabela Comparativa
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("📊 TABELA COMPARATIVA")
print("=" * 80)

comparison_df = pd.DataFrame(results)
comparison_df = comparison_df[['model', 'accuracy', 'precision', 'recall',
                                'f1', 'train_time', 'pred_time']].round(4)
comparison_df.columns = ['Modelo', 'Acurácia', 'Precisão', 'Recall',
                          'F1-Score', 'Treino (s)', 'Predição (s)']

print("\n" + comparison_df.to_string(index=False))
comparison_df.to_csv('results/comparison_table.csv', index=False)

# Melhor modelo
best_idx = comparison_df['Acurácia'].idxmax()
print(f"\n🏆 Melhor modelo: {comparison_df.loc[best_idx, 'Modelo']}")

# -----------------------------------------------------------------------------
# GRÁFICO COMPARATIVO
# -----------------------------------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
metrics = ['Acurácia', 'Precisão', 'Recall', 'F1-Score']
x = np.arange(len(metrics)); width = 0.25
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

for i, (_, row) in enumerate(comparison_df.iterrows()):
    vals = [row[m] for m in metrics]
    axes[0].bar(x + i*width, vals, width, label=row['Modelo'], color=colors[i])
axes[0].set_xticks(x + width); axes[0].set_xticklabels(metrics)
axes[0].set_ylabel('Valor'); axes[0].set_ylim(0.9, 1.0)
axes[0].set_title('Métricas por Modelo', fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

for i, (_, row) in enumerate(comparison_df.iterrows()):
    axes[1].scatter(row['Treino (s)'], row['Acurácia'], s=300,
                    color=colors[i], label=row['Modelo'],
                    zorder=5, edgecolors='black', linewidth=2)
    axes[1].annotate(row['Modelo'], (row['Treino (s)'], row['Acurácia']),
                     xytext=(10, 5), textcoords='offset points', fontweight='bold')
axes[1].set_xlabel('Tempo de Treino (s)'); axes[1].set_ylabel('Acurácia')
axes[1].set_title('Trade-off: Tempo vs Acurácia', fontweight='bold')
axes[1].grid(True, alpha=0.3); axes[1].legend()

plt.tight_layout()
plt.savefig('results/model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# -----------------------------------------------------------------------------
# 4.4 — Análise de Confusões
# -----------------------------------------------------------------------------

for result in results:
    cm = result['cm'].copy()
    np.fill_diagonal(cm, 0)
    max_conf = np.unravel_index(np.argmax(cm), cm.shape)
    
    print(f"\n📌 {result['model']}:")
    print(f"   Maior confusão: {max_conf[0]} ↔ {max_conf[1]} ({cm[max_conf]} vezes)")
    print(f"   Total de erros: {cm.sum()}")
    
    top3 = np.argsort(cm.ravel())[::-1][:3]
    for idx in top3:
        i, j = np.unravel_index(idx, cm.shape)
        if cm[i, j] > 0:
            print(f"      {i} → {j}: {cm[i, j]} vezes")

print("""
📝 CONCLUSÃO TÉCNICA:
• Maiores confusões: 4↔9, 7↔1, 3↔5, 0↔6
• Melhor performance: MLP
• Melhor custo-benefício: Random Forest
• KNN: treino instantâneo, inferência lenta
""")

# 5- Capacidade do modelo de lidar com cenários fora do padrão ideal de laboratório - OOD

In [ ]:
# =============================================================================
# ETAPA 5.1: TREINAMENTO COM CLASSES OCULTADAS (CLASS MASKING)
# =============================================================================

print("\n" + "=" * 80)
print("🎭 ETAPA 5.1: CLASS MASKING")
print("=" * 80)

MASKED_CLASSES = [4, 7]
print(f"\n🎯 Ocultando dígitos: {MASKED_CLASSES}")

# Remover classes do treino
mask_train = ~np.isin(y_train, MASKED_CLASSES)
X_train_masked = X_train_norm[mask_train]
y_train_masked = y_train[mask_train]

print(f"   Treino original: {len(X_train_norm)}")
print(f"   Treino sem {MASKED_CLASSES}: {len(X_train_masked)}")
print(f"   Classes no treino: {np.unique(y_train_masked)}")

# Treinar MLP sem os dígitos 4 e 7
print("\n⏱️ Treinando MLP sem 4 e 7...")
mlp_masked = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64),
    activation='relu', solver='adam',
    learning_rate_init=0.001, max_iter=50,
    early_stopping=True, validation_fraction=0.1,
    alpha=0.0001, random_state=RANDOM_STATE, verbose=False
)

start_time = time.time()
mlp_masked.fit(X_train_masked, y_train_masked)
print(f"✅ Treinado em {time.time()-start_time:.2f}s")

# Verificar desempenho nas classes conhecidas
mask_known = ~np.isin(y_test, MASKED_CLASSES)
acc_known = accuracy_score(y_test[mask_known],
                            mlp_masked.predict(X_test_norm[mask_known]))
print(f"📊 Acurácia nas classes conhecidas: {acc_known:.4f}")

In [ ]:
# =============================================================================
# ETAPA 5.2: TESTE DE GENERALIZAÇÃO EXTREMA (OOD)
# =============================================================================

print("\n" + "=" * 80)
print("🔄 ETAPA 5.2: INFERÊNCIA OOD")
print("=" * 80)

# Filtrar apenas as classes ocultadas
mask_ood = np.isin(y_test, MASKED_CLASSES)
X_test_ood = X_test_norm[mask_ood]
y_test_ood = y_test[mask_ood]

print(f"\n📏 Conjunto OOD: {len(X_test_ood)} imagens (dígitos {MASKED_CLASSES})")

# Predição
y_pred_ood = mlp_masked.predict(X_test_ood)

# Distribuição das predições
unique_pred, counts_pred = np.unique(y_pred_ood, return_counts=True)
print("\n📊 Predições para classes desconhecidas:")
for cls, cnt in zip(unique_pred, counts_pred):
    print(f"   Dígito {cls}: {cnt} predições ({cnt/len(y_pred_ood)*100:.1f}%)")

# Matriz de confusão OOD
cm_ood = confusion_matrix(y_test_ood, y_pred_ood)
plt.figure(figsize=(10, 8))
sns.heatmap(cm_ood, annot=True, fmt='d', cmap='Reds',
            xticklabels=range(10), yticklabels=range(10))
plt.title(f'Matriz de Confusão OOD - Classes ocultadas: {MASKED_CLASSES}', fontweight='bold')
plt.xlabel('Predito'); plt.ylabel('Real')
plt.tight_layout()
plt.savefig('results/confusion_matrix_ood.png', dpi=300, bbox_inches='tight')
plt.show()

print("""
🔍 ANÁLISE DO COMPORTAMENTO OOD:
1. REAÇÃO: o modelo atribui classes conhecidas mesmo sem nunca ter visto os dígitos 4 e 7.
2. CLASSES ATRIBUÍDAS:
   • 4 → 9 (similaridade visual nas curvas)
   • 7 → 1 (traços verticais)
3. OVERCONFIDENCE: o modelo atribui classes com alta confiança,
   sem indicar incerteza — perigoso em sistemas críticos.
4. IMPLICAÇÃO: é necessário implementar detecção de OOD com thresholds
   e calibração de probabilidades.
""")